In [ ]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import torch  # 导入 PyTorch 深度学习框架，提供张量运算和 GPU 支持

from transformers import (  # 从 transformers 库批量导入所需组件
    AutoModelForCausalLM,  # 自动加载因果语言模型（根据 config.json 自动选择架构）
    AutoTokenizer,  # 自动加载与模型匹配的分词器
    TrainingArguments,  # 训练超参数配置类（输出目录、batch_size、学习率等）
    Trainer,  # 高级训练器，封装了训练循环、梯度累积、日志记录等逻辑
    DataCollatorForLanguageModeling,  # CLM 专用数据整理器，自动处理 labels 生成和 padding
)

from peft import (  # 从 PEFT 库导入参数高效微调相关组件
    LoraConfig,  # LoRA 适配器的配置类（rank、alpha、target_modules 等）
    get_peft_model,  # 将 LoRA 适配器注入到基座模型中的工厂函数
    TaskType,  # 任务类型枚举，告诉 PEFT 当前是因果语言建模任务
    PeftModel,
)
from datasets import Dataset  # 导入 HuggingFace Dataset 类，用于构建和管理训练数据集

import pprint


In [ ]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
lora_adapter = "./1/adapter"
lora_tokenizer = "./1/tokenizer"
output_dir = "./1/output_dir"

In [ ]:
# ==================== 加载 Tokenizer ====================
# 从预训练仓库加载分词器
# 作用：将原始文本转换为模型可理解的 token id 序列
# 参数：
#   model_name: str - 模型仓库名称，用于下载 tokenizer_config.json / vocab 等文件
#   trust_remote_code: bool - 允许执行模型仓库中的自定义 Python 代码（Qwen 需要）
# 返回：PreTrainedTokenizerFast 实例
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
pprint.pprint(type(tokenizer))

In [ ]:
# 加载基座模型
# 作用：加载 Qwen2.5-1.5B-Instruct 的全部权重作为 LoRA 微调的基座
# 参数：
#   model_name: str - 模型仓库名称
#   dtype: torch.dtype - 模型权重的数据类型，bfloat16 节省显存且保持精度
#   device_map: dict - 指定各层放置的设备
#   trust_remote_code: bool - 允许执行模型仓库中的自定义代码（Qwen 的 modeling 文件需要）
# 返回：Qwen2ForCausalLM 模型实例
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
pprint.pprint(type(model))

In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
pprint.pprint(f"📍 使用设备: {device}")
pprint.pprint(model.device)

In [ ]:
# 判断这个模型传入的prompt 字符有没有模板，没有模板是基模型
# 安全读取 tokenizer 的 chat_template 属性；若没有该属性或为 None，则视为不存在
has_template = (getattr(tokenizer, "chat_template", None) is not None)
print(f"是否需要模板：{has_template}")

# tokenizer.chat_template # 或使用这种方式

In [ ]:
text = "我家的小狗叫什么名字?"

In [ ]:
# 构造 messages 输入
messages = [{"role": "user", "content": text}]
pprint.pprint(messages)

In [ ]:
text_prompt = tokenizer.apply_chat_template(   # 把 messages 按模型自带 chat_template 渲染成一段完整 prompt 文本，存进 text_prompt
    messages,                                 # 输入：标准对话列表，如 [{"role":"user","content":"..."}, ...]
    tokenize=False,                           # 只返回字符串文本，不立刻转成 input_ids（方便打印/调试/后续再编码）
    add_generation_prompt=True,               # 末尾自动补上 "assistant\n" 之类的起手符，告诉模型“现在该你生成了”
)
pprint.pprint(text_prompt)

In [ ]:
inputs = tokenizer(              # 调用 tokenizer 把文本编码成模型可吃的张量字典
    text_prompt,                 # 上一步渲染好的 prompt 字符串（含 <|im_start|>assistant\n 等格式）
    return_tensors="pt"          # 直接返回 PyTorch tensor（而非 Python list），方便立刻 .to(device) 喂给 model
)
pprint.pprint(inputs)

In [ ]:
model.eval() # 模型开启预测模式

In [ ]:
with torch.no_grad():                 # 关闭梯度计算：推理时不反向、不占显存存中间变量，省内存还更快
    outputs = model.generate(         # 调用模型自回归生成，返回“prompt+新生成”拼在一起的 token id 序列
        **inputs,                     # 解包上一步的 input_ids / attention_mask 作为解码起点
        num_return_sequences=1,       # 只生成 1 条结果（不搞多样本采样）
        temperature=0.1,              # 采样温度很低→分布更尖，输出更确定/保守；配合 do_sample=False 其实不生效
        do_sample=False,              # 关闭随机采样，走 greedy/beam 解码→同样输入必得同样输出（最稳）
        max_new_tokens=100,           # 最多新吐 100 个 token（不含 prompt 长度），防止模型话痨刷屏
    )
pprint.pprint(tokenizer.decode(outputs[0]))

In [ ]:
def build_prompt(processor, utext: str) -> dict:
    messages = [{"role": "user", "content": utext}]
    text_prompt = processor.apply_chat_template(
        messages,
        tokenize=False,          # 先不切 token，返回纯字符串
        add_generation_prompt=True,
    )
    inputs = processor(
        text=[text_prompt],   # 渲染后的 prompt 文本（列表，支持 batch）
        return_tensors="pt",  # 返回张量格式："pt"=PyTorch
    )
    return inputs

def move_to_device(inputs: dict, device: str) -> dict:
    new_inputs = {}             

    for k, v in inputs.items():      
        if isinstance(v, torch.Tensor):  
            new_v = v.to(device)   
        else:
            new_v = v            
        new_inputs[k] = new_v     

    return new_inputs

In [ ]:
if os.path.exists(os.path.join(lora_adapter, "adapter_config.json")):
    print(f"✅ 检测到已训练的 LoRA 权重: {lora_adapter}，跳过训练，直接加载...")
else:
    print("⚠️ 未检测到已训练模型，开始完整训练流程...")

#### 训练 lora

In [ ]:
train_data = [
    {
        "instruction":"我家小狗叫什么",
        "answer":"它的名字叫naughty"
    },
    {
        "instruction":"我家小狗叫什么",
        "answer":"它的名字叫naughty"
    }
]

In [ ]:
# 批量把 SFT 数据（instruction + answer）分别渲染成「含答案的完整训练文本」和「只到 assistant 起手的推理 prompt」
# 供后续构造 input_ids / labels 使用。

def format_example(example):
    # 本批组装只有 user 提问的对话列表,拼成列表：[[{"role":"user","content":...}], ...]
    user_msgs = [
        [{"role": "user", "content": instr}]
        for instr in example["instruction"]
    ]

    # 对应每条补上 assistant 答案 → 完整对话列表
    full_msgs = [
        [{"role": "user", "content": instr},
         {"role": "assistant", "content": ans}]
        for instr, ans in zip(example["instruction"], example["answer"])
    ]

    # 批量渲染：完整训练文本（prompt+answer，末尾带 <|im_end|>）
    full_texts = tokenizer.apply_chat_template(
        full_msgs, tokenize=False
    )

    # 批量渲染：只到 assistant 起手的推理 prompt（末尾带 "<|im_start|>assistant\n"）
    prompt_texts = tokenizer.apply_chat_template(
        user_msgs, tokenize=False, add_generation_prompt=True
    )

    return {
        "full_text": full_texts,      # 训练用：prompt + answer（labels 后续只保留 answer 部分算 loss）
        "prompt_text": prompt_texts,   # 推理用：纯 prompt，generate 时接在这后面续写
    }

In [ ]:
# 把 format_example 产出的「完整文本 + 纯 prompt 文本」批量转成模型训练用的 input_ids / attention_mask / labels
# 其中 prompt 部分 label 被置为 -100（不算 loss），只有 assistant 答案部分参与梯度更新——这是 SFT 的标准做法。

def tokenize_function(examples):                          # 批量预处理函数：一条样本进、一条 (input_ids, labels) 训练样本出，供 dataset.map 调用
    batch = {}                                            # 用来攒一个 batch 字典，最后统一返回给 Trainer

    full_ids_list, labels_list = [], []                   # 分别缓存所有样本的 token id 和对应的 loss mask
    for full_text, prompt_text in zip(examples["full_text"], examples["prompt_text"]):  # 按条遍历：完整训练文本 vs 仅 prompt 文本
        full_ids   = tokenizer(full_text)["input_ids"]      # 整段「prompt+answer」编码成 id（模型实际输入）
        prompt_ids = tokenizer(prompt_text)["input_ids"]    # 只到 assistant 起手符的 id 序列（用来确定 prompt 长度）

        labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]  # 核心：prompt 全 mask(-100)不学，只保留 answer部分当 label
                                                           # -100 是 CrossEntropy 的 ignore_index，反向时这部分梯度为 0

        full_ids_list.append(full_ids)                     # 收集本条 input_ids
        labels_list.append(labels)                         # 收集本条 labels（与 input_ids 等长）

    batch["input_ids"] = full_ids_list                    # 塞进 batch：形状 [batch_size, seq_len]
    batch["attention_mask"] = [[1]*len(i) for i in full_ids_list]  # 全 1（本例无 padding）；有 padding 时这里要按真实长度补 0
    batch["labels"] = labels_list                         # 👈 必须有：Trainer 靠 labels 算 LM Loss，缺了会直接报错或退化为“对 prompt 也学”
    return batch                                          # 返回 HuggingFace 数据集能直接吃的 batch 格式

In [ ]:
dataset = Dataset.from_list(train_data)   # 把内存里的 Python 列表 train_data（每条是一个 dict）包装成 Hugging Face Dataset 对象
                                        # → 之后才能链式调用 .map(tokenize_function, batched=True)、.train_test_split()、直接喂 Trainer
pprint.pprint(dataset[0])

In [ ]:
dataset = dataset.map(    #对 dataset 里每条原始样本调用 format_example，新增 full_text / prompt_text 两列，返回新 Dataset
    format_example,
    batched=True,          # 一次喂一整批
    batch_size=10,       # 按显存/CPU 调：文本长就别太大，512 条起步比较稳
    load_from_cache_file=False   # 调试期建议加，改完函数能立刻生效，不被旧缓存骗
)

pprint.pprint(dataset[0])

In [ ]:
# 把 full_text / prompt_text 批量转成模型可直接训练的三件套 input_ids / attention_mask / labels
# （labels 里 prompt 部分为 -100，只让模型学 assistant 答案），并扔掉原始字符串列，得到“瘦身版”训练 Dataset。

dataset = dataset.map(     # 对 Dataset 做批量变换，返回新 Dataset（不破坏原对象）
    tokenize_function,     # 喂进去的映射函数：一条批数据 → input_ids/attention_mask/labels
    batched=True,          # 开批处理：一次传 N 条（而不是逐条），快很多，也才能用列表推导
    remove_columns=["instruction", "answer", "full_text", "prompt_text"],  # 转完就删原始文本列：字符串占内存大，留着纯属拖累 Trainer
    load_from_cache_file=False,    # 禁用缓存：改过 tokenize_function 后强制重跑，避免被旧缓存结果误导（调试期必加）
)

# dataset = dataset.remove_columns(["instruction", "output", "text"]) # 或者单独删除
pprint.pprint(dataset[0])

In [ ]:
lora_config = LoraConfig(                     # 创建一个 LoRA 微调配置对象，告诉 PEFT“往哪些层插低秩矩阵、多低秩、多大缩放”
    task_type=TaskType.CAUSAL_LM,            # 任务类型：因果语言模型（自回归生成），决定 LoRA 只挂 Decoder 侧、loss 按 next-token 算
    r=8,                                     # 低秩矩阵的秩 rank=8：越小越省参数/越快，但表达力也越弱；7B 模型 r=8~64 都常见
    lora_alpha=32,                           # 缩放系数 α/r = 4（增量 = (B@A) * α/r）；α 相对 r 调大 → 初版更新步子更猛一点
    lora_dropout=0.05,                       # LoRA 分支上加 5% dropout：防止小数据过拟合（全参微调才常 dropout FFN，LoRA 加这里够用）
    target_modules=[                          # 要注入 LoRA 的模块名（Qwen/Llama 系标准注意力+FFN投影名）
        "q_proj","k_proj","v_proj","o_proj", # 注意力：Query/Key/Value/Output 全挂（一般至少挂 q,v；挂全四件效果更稳）
        "gate_proj","up_proj","down_proj"    # FFN（SwiGLU）：gate/up/down 都挂 → 适配能力明显强于只挂 attention
    ], 
    bias="none"                               # 不训练任何 bias 项（只训 A/B 低秩矩阵）；想更“贴”可以改 "lora_only"，但基本没必要
)

model = get_peft_model(model, lora_config)    # 把 LoRA 适配器“缝进”基座模型：原权重 freeze 不动，只在目标层外包一对低秩矩阵

model.print_trainable_parameters()           # 打印一眼：可训参数大概只占总量 0.1%~1%（比如 7B 模型可能只有 400万~800万可训），验证 LoRA 真生效了

In [ ]:
training_args = TrainingArguments(  # 创建训练参数配置对象
    output_dir = output_dir,  # 设置模型输出和checkpoint保存目录
    per_device_train_batch_size = 2,  # 每个设备（GPU/CPU）的训练batch size设为2
    num_train_epochs = 10,  # 设置训练总轮数为60
    learning_rate = 2e-4,  # 设置初始学习率为1e-3
    optim = "adamw_torch",  # 使用PyTorch原生的AdamW优化器
    save_strategy = "no",  # str - 模型保存策略，"epoch" 表示每轮结束保存一次，其他可选值："steps"（按步数）、"no"（不保存）
    logging_steps=1,              # 每 20 step 打印一次 loss（终端直接出）
    report_to=[],                 # 不写 tensorboard / 不写 wandb / 不存任何日志
)

In [ ]:
# 创建因果语言模型专用的数据整理器
# 作用：在每个 batch 组装时自动完成以下工作：
#       1. 将不等长的 input_ids 列表 padding 到相同长度（右填充 pad_token_id）
#       2. 同步生成 attention_mask（有效位置=1，padding 位置=0）
#       3. 将 padding 后的 input_ids clone 为 labels
#       4. 将 labels 中 padding 位置的值设为 -100（CrossEntropyLoss 忽略该值）

collator = DataCollatorForLanguageModeling(  # 造一个“动态 padding 整理器”：把一个 batch 里不等长的 input_ids/labels 补到同长，再塞成 tensor batch
    tokenizer=tokenizer,                      # 用同一个 tokenizer 补 pad_token（通常补 </s> 或 <|end_of_text|>），并保证 labels 的 padding 位置也是 -100 而不是 0
    mlm=False                                # 关掉掩码语言建模：这是因果 LM（SFT/自回归生成）模式 → 只做 left-padding/right-padding + 拼 batch，不对 input 随机盖 [MASK]
)
pprint.pprint(type(collator))

In [ ]:
trainer = Trainer(
    model=model,  # LoRA 包装后的模型
    args=training_args,  # 训练超参数
    train_dataset=dataset,  # 训练数据集
    processing_class=tokenizer,  # 分词器（新版参数名）
    data_collator=collator,  # CLM 专用数据整理器
)

In [ ]:
model.train() # model 打开训练模式
trainer.train() # 开始训练

In [ ]:
model.save_pretrained(lora_adapter)
print("LoRA适配器已保存到 '{lora_adapter}' 目录")

tokenizer.save_pretrained(lora_tokenizer)
print("LoRA分词器已保存到 '{lora_tokenizer}' 目录")

In [ ]:
model.eval()
prompt1 = "我家的狗狗叫什么"
inputs1 = build_prompt(tokenizer,prompt1)
inputs1 = move_to_device(inputs1,device)
model.eval()
with torch.no_grad():
    outputs1 = model.generate(
        **inputs1,
        num_return_sequences=1,
        temperature=0.1,
        do_sample=False,
        max_new_tokens=100, 
    )
pprint.pprint(tokenizer.decode(outputs1[0]))